# Prompt Engineering for Image Generation

In [ ]:
from IPython.display import Image, Markdown, display
from google import genai
from google.genai import types

In [ ]:
import os
from helper import authenticate

credentials, project_id = authenticate()
client = genai.Client(
    project=project_id,
    location="global",
    credentials=credentials,
    http_options=types.HttpOptions(
         base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
    )
)

In [ ]:
IMAGE_MODEL_ID = "gemini-3.1-flash-image-preview"
TEXT_MODEL_ID = "gemini-3-flash-preview"

## Simple prompt

In [ ]:
prompt = """
A diagram illustrating visual proof of the Pythagorean theorem.
"""

response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=types.ImageConfig(
            # aspect ratios: 1:1, 3:2, 2:3, 3:4, 4:3, 1:4, 4:1, 4:5, 5:4, 1:8, 8:1, 9:16, 16:9, 21:9
            aspect_ratio="16:9",
        ),
    ),

)

for part in response.candidates[0].content.parts:
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

## Prompt enhancement

In [ ]:
subject = "a simple slide"
action = "explaining visual proof of the Pythagorean theorem"
location = "white background"
camera_control = "eye-level shot"
lighting = "white light"
style = "minimalist"

In [ ]:
reference_image = "slide-template.png"
display(Image(filename=reference_image, width=500))

In [ ]:
keywords = [subject, action, location, camera_control, lighting, style]

gemini_prompt = f"""
Your task is to expand the following keywords into a single, high-fidelity, 
descriptive prompt for image generation. Every single keyword MUST be 
included. Include reference images if provided and use that image as a 
reference style guide for generated images. Output ONLY the final prompt 
string, without any introduction or explanation. Mandatory Keywords: 
{",".join(keywords)}
"""

response = client.models.generate_content(
    model=TEXT_MODEL_ID,
    contents=gemini_prompt,
)

image_prompt = response.text
display(Markdown(response.text))

In [ ]:
with open(reference_image, "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=image,
            mime_type="image/png",
        ),
        f"""Use this reference image as a general style guide to generate a lecture 
        slide. Include only the necessary sections from the template. Use the guide 
        as a general style formatting template instead of a strict outline. The 
        slide image should be the entire image. Slide concept: {image_prompt}""",
    ],
    config=types.GenerateContentConfig(
        response_modalities=['IMAGE'],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    ),

)

for part in response.candidates[0].content.parts:
    if part.inline_data:
        image_data=part.inline_data.data
        display(Image(data=image_data, width=500))

with open("slide-image.png", "wb") as image_file:
    image_file.write(image_data)